# 01 — Pré-processamento

Este notebook transforma os CSVs em `data/raw/` em uma única base mensal tratada. A lógica de transformação mora em `src/cancer_covid/data.py`; aqui ficam apenas a execução e a verificação.

**Regra importante:** `-` em `U07.1` é interpretado como ausência de registro e convertido para zero.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cancer_covid.config import PROCESSED_FILE, RAW_DATA_DIR
from cancer_covid.data import build_processed_dataset


## 1. Construir a base tratada

A rotina valida colunas, datas, duplicidades e valores negativos antes de salvar o resultado.

In [2]:
dados = build_processed_dataset(RAW_DATA_DIR, PROCESSED_FILE)
print(f'Arquivo gerado: {PROCESSED_FILE.relative_to(PROJECT_ROOT)}')
dados.head()


Arquivo gerado: data/processed/cancer_covid_mensal.csv


,month_year,C34,C91,C92,U07.1,city,year,month,total_cancers
0,2018-01-01,5,1,2,0,Campinas,2018,1,8
1,2018-02-01,3,0,1,0,Campinas,2018,2,4
2,2018-03-01,7,0,2,0,Campinas,2018,3,9
3,2018-04-01,4,0,2,0,Campinas,2018,4,6
4,2018-05-01,5,1,1,0,Campinas,2018,5,7


## 2. Verificações finais

Cada linha deve representar um município e mês; as contagens não podem ser negativas.

In [3]:
assert not dados.duplicated(['city', 'month_year']).any()
assert (dados[['C34', 'C91', 'C92', 'U07.1']] >= 0).all().all()
dados.groupby('city').agg(inicio=('month_year', 'min'), fim=('month_year', 'max'), meses=('month_year', 'size'))


,inicio,fim,meses
city,,,
Campinas,2018-01-01,2023-03-01,63
Guarulhos,2018-01-01,2023-03-01,63
São Paulo,2018-01-01,2023-03-01,63
